<a href="https://colab.research.google.com/github/erdijova/Introduction-to-Machine-Learning-with-Python-A-Guide-for-Data-Scientists/blob/main/8_wrapping_up.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Wrapping Up

At this point, you have learned how to apply key machine learning methods for both supervised and unsupervised tasks, enabling you to solve a wide range of problems.

This final section does not introduce new techniques. Instead, it focuses on practical insights and lessons that are essential for becoming effective in real-world data science.

The goal is to connect everything you’ve learned and highlight the mindset and best practices that distinguish strong practitioners from those who only understand the theory.

## Approaching a Machine Learning Problem

Machine learning is only one part of a bigger process, so don’t start with algorithms right away.

### Start with the Question, Not the Algorithm

First, clearly define your goal:

- **Exploratory analysis**: find patterns or insights in data  
- **Predictive modeling**: predict outcomes from input data  
- **Causal analysis**: understand cause-and-effect relationships  

### Define Success Before Building Models

Before building a model, decide how success will be measured and whether the problem is worth solving.

Key points:
- Use a metric that reflects real-world impact  
- Make sure your data is relevant and reliable  
- Consider that data patterns can change over time  
- Estimate the potential value of solving the problem  

**Rule of thumb:** if the solution doesn’t create meaningful value, it’s not worth the effort.

### The Iterative Workflow

Machine learning is not a one-way process. It works as a continuous cycle:

Collect data → Clean data → Build model → Analyze errors → Improve data → repeat

One of the most important steps is analyzing model errors. Mistakes reveal what the model is missing and guide improvements.

For example, if a model fails on certain patterns, it may need:
- better features
- more relevant data
- or a revised problem setup

A key insight:
Improving data and features usually has a bigger impact than endlessly tuning model parameters.

In practice:
- Beginners focus on algorithms
- Experienced practitioners focus on data quality, features, and problem definition

## Humans in the Loop

You should also consider if and how to have humans in the loop. Some processes (like pedestrian detection in a self-driving car) need to make **immediate decisions** with no time for human review. Others might not need immediate responses, so it can be possible to have humans confirm uncertain decisions.

### The Confidence-Based Routing Pattern

A powerful design pattern is to use the model's **prediction confidence** to route decisions:

$$\text{Decision} = \begin{cases} \text{Auto-approve} & \text{if } P(\hat{y} \mid \mathbf{x}) > \tau_{\text{high}} \\ \text{Human review} & \text{if } \tau_{\text{low}} \leq P(\hat{y} \mid \mathbf{x}) \leq \tau_{\text{high}} \\ \text{Auto-reject} & \text{if } P(\hat{y} \mid \mathbf{x}) < \tau_{\text{low}} \end{cases}$$

where $\tau_{\text{high}}$ and $\tau_{\text{low}}$ are confidence thresholds tuned to the application's precision and recall requirements.

Many applications are dominated by "simple cases" for which an algorithm can make a confident decision, with relatively few "complicated cases" that get routed to a human. Even automating just 50% or 10% of decisions can significantly reduce cost and response time. In medical applications, for example, an algorithm might flag suspicious X-rays for radiologist review rather than attempting to make the final diagnosis autonomously -- the human provides the precision, while the machine provides the screening capacity.

## From Prototype to Production

Machine learning tools are great for building quick prototypes, but production systems have different challenges.

In real-world systems, models may need to be integrated into larger infrastructures, sometimes requiring reimplementation in faster or more scalable languages.

### Production Requirements

Production models must meet several key requirements:

- **Reliability**: handle unexpected inputs without crashing  
- **Latency**: produce predictions quickly (often in milliseconds)  
- **Memory**: fit within system or device constraints  
- **Maintainability**: remain simple enough to manage over time  

Each added component increases system complexity and maintenance cost.

### Key Insight

Simple models are often better in production. A smaller, well-designed model can be more reliable and efficient than a complex one with only slightly better accuracy.

Always balance performance with simplicity, robustness, and long-term maintainability.

## Testing Production Systems

Evaluating a model on a test set (offline evaluation) is important, but not enough for real-world systems.

### A/B Testing

To truly measure impact, we use **online testing**, where the model is evaluated in real usage.

A common method is **A/B testing**:
- **Group A**: users see the new model
- **Group B**: users see the current system

We then compare key metrics (such as clicks, engagement, or revenue) between the two groups.

This helps detect real-world effects that may not appear in offline testing. For example, a model that looks better in theory might actually reduce user satisfaction.

### Key Insight

User behavior can be unpredictable, so testing in a live environment is essential before fully deploying a model.

More advanced approaches, like adaptive testing methods, can further optimize how experiments are run.

## Building Your Own Estimator

The algorithms provided by scikit-learn cover a wide range of tasks. However, often there will be some particular processing you need for your data that is not implemented in scikit-learn. As we discussed in Chapter 6, all data-dependent processing should be inside the cross-validation loop, which means your custom processing needs to be compatible with the scikit-learn interface.

The solution: **build your own estimator!** Implementing a transformer or estimator compatible with `Pipeline`, `GridSearchCV`, and `cross_val_score` is surprisingly easy.

In [1]:
from sklearn.base import BaseEstimator, TransformerMixin

class MyTransformer(BaseEstimator, TransformerMixin):
    def __init__(self, first_parameter=1, second_parameter=2):
        # All parameters must be specified in the __init__ function
        self.first_parameter = first_parameter
        self.second_parameter = second_parameter

    def fit(self, X, y=None):
        # fit should only take X and y as parameters
        # Even if your model is unsupervised, you need to accept a y argument!
        # Model fitting code goes here
        print("fitting the model right here")
        # fit returns self
        return self

    def transform(self, X):
        # transform takes as parameter only X
        # Apply some transformation to X
        X_transformed = X + 1
        return X_transformed

By inheriting from `BaseEstimator` and `TransformerMixin`, our custom class automatically gets:

**From `BaseEstimator`:** The `get_params()` and `set_params()` methods, which are essential for `GridSearchCV` to tune your transformer's parameters. These methods work by inspecting the `__init__` signature, which is why all parameters must be explicitly listed there.

**From `TransformerMixin`:** The `fit_transform()` method, which is a convenience method that calls `fit(X, y)` followed by `transform(X)`. This is important for pipelines where `fit_transform` is called during training.

Let's verify that our custom transformer works with scikit-learn's tools:

In [2]:
import numpy as np
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

X = np.array([[1, 2], [3, 4], [5, 6]])

# Use in a pipeline
pipe = Pipeline([
    ('my_transform', MyTransformer()),
    ('scaler', StandardScaler())
])

result = pipe.fit_transform(X)
print("Original X:\n", X)
print("\nAfter MyTransformer (+1) then StandardScaler:\n", result)
print("\nMyTransformer params:", MyTransformer().get_params())

fitting the model right here
Original X:
 [[1 2]
 [3 4]
 [5 6]]

After MyTransformer (+1) then StandardScaler:
 [[-1.22474487 -1.22474487]
 [ 0.          0.        ]
 [ 1.22474487  1.22474487]]

MyTransformer params: {'first_parameter': 1, 'second_parameter': 2}


Our custom transformer works seamlessly in a `Pipeline`. The data flows through `MyTransformer` (which adds 1 to all values), then through `StandardScaler` (which standardizes each feature to zero mean and unit variance). The `get_params()` method correctly returns the parameters we defined in `__init__`, which means `GridSearchCV` can tune them:

```python
param_grid = {'my_transform__first_parameter': [1, 2, 3]}
grid = GridSearchCV(pipe, param_grid, cv=5)
```

The key rules for building compatible estimators are:

**For transformers** (preprocessing steps): Inherit from `BaseEstimator` and `TransformerMixin`, implement `__init__`, `fit(X, y=None)`, and `transform(X)`. The `fit` method must return `self`.

**For classifiers:** Inherit from `BaseEstimator` and `ClassifierMixin`, implement `__init__`, `fit(X, y)`, and `predict(X)`. Optionally implement `predict_proba(X)` for probability estimates.

**For regressors:** Inherit from `BaseEstimator` and `RegressorMixin`, implement `__init__`, `fit(X, y)`, and `predict(X)`.

Most scikit-learn users build up a **collection of custom models** over time, tailored to their specific domains and workflows.

## Where to Go from Here

This book provides an introduction to machine learning and will make you an effective practitioner. Here are suggestions for deepening your skills in specific directions.

### Theory

In this book, we tried to provide intuition for the most common algorithms without requiring a strong mathematics background. However, many of the models use principles from **probability theory**, **linear algebra**, and **optimization**. Knowing the theory behind the algorithms will make you a better data scientist.

Recommended theory books:

**The Elements of Statistical Learning** (Hastie, Tibshirani, Friedman) -- The definitive reference for statistical learning theory. Freely available online. Covers the mathematical foundations of nearly every algorithm we discussed.

**Pattern Recognition and Machine Learning** (Bishop) -- Emphasizes the probabilistic framework. Excellent for understanding Bayesian approaches, graphical models, and the principled treatment of uncertainty.

**Machine Learning: A Probabilistic Perspective** (Murphy) -- A comprehensive 1,000+ page treatment featuring in-depth discussions of state-of-the-art approaches, far beyond what we could cover in this book.

### Other Machine Learning Frameworks and Packages

Depending on your needs, Python and scikit-learn might not be the best fit:

**statsmodels** (Python): For statistical modeling and inference rather than pure prediction. Provides detailed statistical summaries (confidence intervals, $p$-values, diagnostic tests) that scikit-learn intentionally omits.

**R**: Another lingua franca of data scientists. R is designed specifically for statistical analysis and is famous for its visualization capabilities (ggplot2) and highly specialized statistical packages.

**vowpal wabbit (vw)**: A highly optimized C++ library with command-line interface, particularly useful for very large datasets and streaming/online learning.

**spark MLlib**: For distributed machine learning on a cluster. If your data is already on a Hadoop filesystem, this may be the natural choice for scaling.

### Deep Learning

While we touched on neural networks briefly in Chapter 2, this is a rapidly evolving area. The book **Deep Learning** by Goodfellow, Bengio, and Courville (MIT Press) provides a comprehensive introduction. In practice, the dominant frameworks are **PyTorch** and **TensorFlow/Keras**, which provide GPU-accelerated training for deep neural networks including CNNs (for images), RNNs/Transformers (for text and sequences), and many other architectures.

### Ranking, Recommender Systems, and Other Learning Types

Besides classification and clustering, there are many important machine learning tasks:

- **Ranking**: ordering results by relevance (used in search engines)  
- **Recommender systems**: suggesting items based on user behavior  
- **Time series forecasting**: predicting future values (e.g., demand, weather)  
- **Reinforcement learning**: learning decisions through interaction and rewards  
- **Semi/self-supervised learning**: using large unlabeled data to improve learning  

These areas expand machine learning beyond standard prediction tasks.

### Probabilistic Modeling and Inference

Many real-world problems have structure that standard models don’t fully capture.

For example, systems like navigation apps combine multiple sources of information (GPS, sensors, maps). A structured approach can model how these sources relate and decide which ones to trust.

Tools like **:contentReference[oaicite:0]{index=0}** and **:contentReference[oaicite:1]{index=1}** allow you to build custom probabilistic models and perform inference more flexibly.

### Key Insight

When problems become more complex, combining domain knowledge with flexible modeling approaches can lead to much better results than using standard models alone.

### Scaling to Larger Datasets

So far, we assumed data fits in memory. For larger datasets, there are two main approaches:

- **Out-of-core learning**: process data in small chunks and update the model step by step  
- **Distributed computing**: split data across multiple machines and process in parallel  

Tools like **:contentReference[oaicite:0]{index=0}** and **:contentReference[oaicite:1]{index=1}** help handle large-scale data.

### Practical Tip

Large infrastructure is not always necessary. In many cases, using a well-chosen subset of the data can achieve similar performance with much lower cost.

### Key Insight

Before scaling up systems, first check if smarter data sampling can solve the problem more efficiently.